In [ ]:
import os

# Set Neo4j connection environment variables
os.environ['NEO4J_URI'] = '<your_neo4j_uri>'
os.environ['NEO4J_USERNAME'] = '<your_username>'
os.environ['NEO4J_PASSWORD'] = '<your_password>'

In [ ]:
from langchain_community.graphs import Neo4jGraph

# Initialize the graph connection
graph = Neo4jGraph(
    url=os.environ['NEO4J_URI'],
    username=os.environ['NEO4J_USERNAME'],
    password=os.environ['NEO4J_PASSWORD']
)

In [ ]:
movie_query = '''
LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/neo4j-examples/movies-csv/master/movies.csv' AS row
MERGE (m:Movie {id: row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdb_rating = toFloat(row.imdbRating)

WITH m, row

// Create director nodes and relationships
FOREACH (director IN split(row.directors, '|') |
  MERGE (p:Person {name: trim(director)})
  MERGE (p)-[:DIRECTED]->(m)
)

// Create actor nodes and relationships
FOREACH (actor IN split(row.actors, '|') |
  MERGE (p:Person {name: trim(actor)})
  MERGE (p)-[:ACTED_IN]->(m)
)

// Create genre nodes and relationships
FOREACH (genre IN split(row.genres, '|') |
  MERGE (g:Genre {name: trim(genre)})
  MERGE (m)-[:IN_GENRE]->(g)
)
'''

In [ ]:
movie_query = '''
LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/neo4j-examples/movies-csv/master/movies.csv' AS row
MERGE (m:Movie {id: row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdb_rating = toFloat(row.imdbRating)

WITH m, row

// Create director nodes and relationships
FOREACH (director IN split(row.directors, '|') |
  MERGE (p:Person {name: trim(director)})
  MERGE (p)-[:DIRECTED]->(m)
)

// Create actor nodes and relationships
FOREACH (actor IN split(row.actors, '|') |
  MERGE (p:Person {name: trim(actor)})
  MERGE (p)-[:ACTED_IN]->(m)
)

// Create genre nodes and relationships
FOREACH (genre IN split(row.genres, '|') |
  MERGE (g:Genre {name: trim(genre)})
  MERGE (m)-[:IN_GENRE]->(g)
)
'''

In [ ]:
result = graph.query(movie_query)
print(result)

In [ ]:
graph.refresh_schema()
print(graph.schema)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
grok_api_key = os.getenv('GROK_API_KEY')

In [ ]:
from langchain_grok import ChatGrok
lm_model = ChatGrok(api_key=grok_api_key, model_name='gamma-2')

In [ ]:
from langchain.chains import GraphCypherQAChain
chain = GraphCypherQAChain.from_llm(graph=graph, llm=lm_model, verbose=True)

In [ ]:
response = chain.invoke({"query": "Who was the director of the movie Casino?"})

In [ ]:
response = chain.invoke({"query": "Who were the actors in the movie Casino?"})